# Predict Students’ Dropout and Academic Success

This notebook uses the same enrollment feature allowlist, data split and folds as the training command in the README. Download the UCI archive before running.

## 1. Allowlist

The decision time is the completion of enrollment, before first-semester teaching and assessment. An explicit allowlist controls which columns reach the models.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)
sns.set_theme(style="whitegrid", context="notebook")

In [ ]:
from student_outcomes.data import (
    CATEGORICAL_FEATURES,
    CLASS_ORDER,
    ENROLLMENT_FEATURES,
    NUMERIC_FEATURES,
    TARGET,
    extract_inputs,
    load_dataset,
    split_dataset,
)

data_path = Path("data/data.csv")
if not data_path.is_file():
    data_path = Path("..") / data_path

df = load_dataset(data_path)
print(f"Loaded {len(df):,} student records and {df.shape[1] - 1} predictors.")

In [ ]:
# Define the enrollment allowlist with a semantic type and availability rationale.
feature_specification = [
    (
        "Marital status",
        "Nominal category",
        "One-hot",
        "Personal background recorded at enrollment",
    ),
    (
        "Application mode",
        "Nominal category",
        "One-hot",
        "Admission route used for this application",
    ),
    (
        "Application order",
        "Ordered category",
        "One-hot",
        "Preference rank submitted with the application",
    ),
    ("Course", "Nominal category", "One-hot", "Degree program selected at enrollment"),
    (
        "Daytime/evening attendance",
        "Binary category",
        "One-hot",
        "Attendance regime selected at enrollment",
    ),
    (
        "Previous qualification",
        "Nominal category",
        "One-hot",
        "Education completed before entry",
    ),
    (
        "Previous qualification (grade)",
        "Numerical measurement",
        "Scale",
        "Grade obtained before entry",
    ),
    (
        "Nacionality",
        "Nominal category",
        "One-hot",
        "Citizenship recorded in the application",
    ),
    (
        "Mother's qualification",
        "Nominal category",
        "One-hot",
        "Family education reported at enrollment",
    ),
    (
        "Father's qualification",
        "Nominal category",
        "One-hot",
        "Family education reported at enrollment",
    ),
    (
        "Mother's occupation",
        "Nominal category",
        "One-hot",
        "Family occupation reported at enrollment",
    ),
    (
        "Father's occupation",
        "Nominal category",
        "One-hot",
        "Family occupation reported at enrollment",
    ),
    (
        "Admission grade",
        "Numerical measurement",
        "Scale",
        "Admission score established before entry",
    ),
    (
        "Displaced",
        "Binary category",
        "One-hot",
        "Relocation status reported at enrollment",
    ),
    (
        "Educational special needs",
        "Binary category",
        "One-hot",
        "Declared support needs at enrollment",
    ),
    (
        "Gender",
        "Binary category",
        "One-hot",
        "Demographic field recorded in the application",
    ),
    (
        "Age at enrollment",
        "Numerical measurement",
        "Scale",
        "Age explicitly measured at enrollment",
    ),
    (
        "International",
        "Binary category",
        "One-hot",
        "International student status at entry",
    ),
]
feature_allowlist = pd.DataFrame(
    feature_specification,
    columns=["Feature", "Semantic type", "Handling", "Availability rationale"],
).set_index("Feature")


ALLOWED_FEATURES = list(ENROLLMENT_FEATURES)
categorical_features = list(CATEGORICAL_FEATURES)
numeric_features = list(NUMERIC_FEATURES)

display(feature_allowlist)
print(f"Allowed predictors = {len(ALLOWED_FEATURES)}")
print(f"Categorical predictors = {len(categorical_features)}")
print(f"Numerical predictors = {len(numeric_features)}")

## 1.1. Excluding future information and uncertain timestamps

All 12 first- and second-semester curricular-unit columns are excluded, including credited and enrolled units. Their released values belong to semester summaries. 

I make an additional conservative modeling choice. UCI supplies no as-of timestamps for `Debtor`, `Tuition fees up to date`, or `Scholarship holder`, and no publication dates for `Unemployment rate`, `Inflation rate`, or `GDP`. These six fields are excluded because their availability at the precise decision time cannot be established from the released file. This is an inference about uncertainty in availability. The choice reflects missing timing evidence.

In [ ]:
# Document exclusions.
SEMESTER_PREFIXES = ("Curricular units 1st sem", "Curricular units 2nd sem")
semester_features = [column for column in df if column.startswith(SEMESTER_PREFIXES)]
uncertain_timing_reasons = {
    "Debtor": "Debt status has no enrollment snapshot timestamp",
    "Tuition fees up to date": "Payment status has no enrollment snapshot timestamp",
    "Scholarship holder": "Scholarship award timing is unspecified",
    "Unemployment rate": "Reference period and publication date are unspecified",
    "Inflation rate": "Reference period and publication date are unspecified",
    "GDP": "Reference period and publication date are unspecified",
}
exclusion_table = pd.DataFrame(
    [
        {"Feature": column, "Reason": "Post-enrollment semester summary"}
        for column in semester_features
    ]
    + [
        {"Feature": column, "Reason": reason}
        for column, reason in uncertain_timing_reasons.items()
    ]
).set_index("Feature")

display(exclusion_table)

In [ ]:
train, holdout = split_dataset(df)
eda_train = train.loc[:, [*ALLOWED_FEATURES, TARGET]].copy()

print(f"Training students available for EDA = {len(train):,}")
print(f"Test students reserved for final evaluation = {len(holdout):,}")

# 2. Exploratory data analysis

In [ ]:
display(
    eda_train[
        [
            "Application mode",
            "Course",
            "Previous qualification",
            "Previous qualification (grade)",
            "Admission grade",
            "Age at enrollment",
            TARGET,
        ]
    ].head()
)

In [ ]:
# Compare documented semantic types with storage dtypes and observed category counts.
feature_type_summary = feature_allowlist[["Semantic type", "Handling"]].copy()
feature_type_summary["Pandas dtype"] = eda_train[ALLOWED_FEATURES].dtypes.astype(str)
feature_type_summary["Unique values"] = eda_train[ALLOWED_FEATURES].nunique()
display(feature_type_summary)

In [ ]:
# Count the three target outcomes and visualize the class imbalance.
class_counts = eda_train[TARGET].value_counts().reindex(CLASS_ORDER)
class_distribution = pd.DataFrame(
    {
        "Students": class_counts,
        "Share, %": class_counts.div(len(eda_train)).mul(100),
    }
)
class_distribution.index.name = TARGET
display(class_distribution.round(2))

class_colors = {"Dropout": "#c95f52", "Enrolled": "#dca84c", "Graduate": "#4b8b83"}
fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(
    CLASS_ORDER, class_counts, color=[class_colors[label] for label in CLASS_ORDER]
)
ax.bar_label(
    bars,
    labels=[f"{count:,} ({count / len(eda_train):.1%})" for count in class_counts],
    padding=5,
)
ax.set(title="Student outcomes on train", ylabel="Students")
ax.set_ylim(0, class_counts.max() * 1.17)
ax.grid(axis="x", visible=False)
plt.tight_layout()
plt.show()

## 2.1. Why accuracy alone is insufficient

The majority rule predicts `Graduate` for every student. On train it gets approximately half of the outcomes right and finds zero `Dropout` or `Enrolled` students. Macro F1 gives each of these outcomes equal weight, which makes it a better selection metric for the three-class task. The final evaluation also looks at precision and recall for `Dropout` because outreach depends on which students are flagged and which are missed.

## 2.2. Numerical distributions on train

Summarize age and both grades on training students. The different units motivate scaling in the logistic regression pipeline. Histograms reveal the observed ranges and concentrations.

In [ ]:
# Summarize and plot numerical enrollment measurements using training students.
display(eda_train[numeric_features].describe().T.round(2))
fig, axes = plt.subplots(1, len(numeric_features), figsize=(15, 4))
for column, ax in zip(numeric_features, axes):
    ax.hist(eda_train[column], bins=30, color="#557b9b", edgecolor="white")
    ax.set(title=column, xlabel="Original value", ylabel="Training students")
plt.tight_layout()
plt.show()

## 2.3. Categories and outcomes on train

Show the frequency of every allowlisted categorical field. Then compare class composition across application choices and attendance regimes. 

In [ ]:
# Summarize the frequency of each categorical feature using training students.
category_frequency_rows = []
for column in categorical_features:
    counts = eda_train[column].value_counts()
    category_frequency_rows.append(
        {
            "Feature": column,
            "Observed categories": len(counts),
            "Most frequent code": counts.index[0],
            "Most frequent share, %": 100 * counts.iloc[0] / len(eda_train),
            "Categories with fewer than 10 students": int(counts.lt(10).sum()),
        }
    )
category_frequency_summary = (
    pd.DataFrame(category_frequency_rows)
    .set_index("Feature")
    .sort_values(by="Most frequent share, %", ascending=False)
)
display(category_frequency_summary.round(2))

In [ ]:
# Compare training class composition by application rank and attendance regime.
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for column, ax in zip(["Application order", "Daytime/evening attendance"], axes):
    counts = pd.crosstab(eda_train[column], eda_train[TARGET]).reindex(
        columns=CLASS_ORDER, fill_value=0
    )
    shares = counts.div(counts.sum(axis=1), axis=0)
    shares.plot.bar(
        stacked=True,
        color=[class_colors[label] for label in CLASS_ORDER],
        ax=ax,
        legend=False,
    )
    ax.set_xticklabels(
        [f"{code}\n(n={total:,})" for code, total in counts.sum(axis=1).items()],
        rotation=0,
    )
    ax.set(
        title=column,
        xlabel="Category code and training group size",
        ylabel="Share within category",
    )
    ax.set_ylim(0, 1)
    ax.grid(axis="x", visible=False)
fig.legend(
    *axes[0].get_legend_handles_labels(),
    loc="upper center",
    bbox_to_anchor=(0.5, 0.94),
    ncol=3,
    frameon=False,
)
fig.suptitle("Training outcome composition by enrollment category", y=0.995)
plt.tight_layout(rect=(0, 0, 1, 0.87))
plt.show()

In [ ]:
# Compare numerical enrollment measurements across the three training outcomes.
fig, axes = plt.subplots(1, len(numeric_features), figsize=(15, 4.5))
for column, ax in zip(numeric_features, axes):
    sns.boxplot(
        data=eda_train,
        x=TARGET,
        y=column,
        order=CLASS_ORDER,
        hue=TARGET,
        hue_order=CLASS_ORDER,
        palette=class_colors,
        legend=False,
        showfliers=False,
        ax=ax,
    )
    ax.set(title=column, xlabel="Outcome")
plt.tight_layout()
plt.show()

display(
    eda_train.groupby(TARGET)[numeric_features]
    .agg(["median", "min", "max"])
    .reindex(CLASS_ORDER)
    .round(2)
)

## 2.4. EDA findings

1. `Graduate` is the largest class on train, followed by `Dropout` and `Enrolled`. The minority class deserves explicit attention in the reports.
2. The grade distributions overlap substantially between outcomes.
3. The training dropout group spans a wider age range.
4. Some application ranks have only one training student, which limits interpretation of their observed outcome shares.

# 3. Experimental protocol

We keep the stratified 80/20 split from the baseline analysis for comparison. All seven models share five shuffled stratified folds from the training set. Each pipeline learns imputation values and encoding categories inside its training fold. Logistic regression also scales the measurements. The majority baseline uses class frequencies alone.

Mean validation macro F1 is the **refit metric**. It gives each outcome one vote. The best configuration in each model family is fitted on all training students. We use the mean and standard deviation of its five validation scores to compare families. These scores reflect configurations chosen on the same folds, and the standard deviation describes variation across folds. The held-out students provide the final evaluation.

In [ ]:
from os import cpu_count

import optuna
from sklearn.model_selection import GridSearchCV

from student_outcomes.models import DEFAULT_TRIALS, fit_models

X_train, y_train_codes = extract_inputs(train)
CV_JOBS = min(4, cpu_count() or 1)

# 4. Model selection

The logistic regression and decision tree use their original parameter grids. Each ensemble gets 24 Optuna trials with a seeded TPE sampler and six initial random trials. Every trial runs on all five folds. The search runs one trial at a time and evaluates up to four folds in parallel, with one thread per estimator. A temporary pipeline cache reuses training-fold preprocessing across configurations.

## 4.1. Ensemble search spaces

| Model | Search space | Rationale |
|:--|:--|:--|
| Random forest | 100–400 trees in steps of 50, depth 4–24, 1–24 students per leaf, 25–100% of features per split, uniform or balanced class weights | The tree count allows averaging across many trees at a bounded cost. Depth and leaf size cover simple and flexible trees. Feature sampling increases diversity and class weights test a more even emphasis on outcomes. |
| Gradient boosting | 60–220 trees in steps of 20, learning rate 0.02–0.20, depth 1–4, 3–30 students per leaf, 65–100% of rows per stage | Shallow trees range from single-feature splits to small interactions. Leaf size and row subsampling reduce the influence of small groups. The smaller tree budget limits the cost of classic boosting. |
| XGBoost | 100–400 trees in steps of 50, learning rate 0.02–0.20, depth 2–7, `min_child_weight` 1–16, 65–100% of rows and features per tree, L2 penalty 0.1–20 | Histogram training supports a wider tree budget. Child weight sets a minimum Hessian sum for a leaf. Sampled rows and columns and the L2 penalty help control complexity. |
| LightGBM | 100–400 trees in steps of 50, learning rate 0.02–0.20, 4–31 leaves, approximately 10–80 students per leaf, 65–100% of features per tree, L2 penalty 0.1–20 | Limiting leaves and the minimum leaf size controls the flexibility of leaf-wise growth on a small dataset. Feature sampling and L2 shrinkage add regularization. |

Learning rates and regularization strengths are sampled on a log scale so the search explores both small and large settings. Smaller learning rates make more gradual updates and can benefit from more boosting stages. Lower bounds on leaf support allow small subgroups while the upper bounds test pooling over more students.

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)
fitted = fit_models(
    X_train, y_train_codes, trials=DEFAULT_TRIALS, jobs=CV_JOBS, extended=True
)

models = {name: model.estimator for name, model in fitted.items()}
cv_scores_by_model = {name: model.fold_scores for name, model in fitted.items()}
best_params_by_model = {name: model.params for name, model in fitted.items()}
baseline_searches = {
    name: model.search
    for name, model in fitted.items()
    if isinstance(model.search, GridSearchCV)
}
studies = {
    name: model.search
    for name, model in fitted.items()
    if isinstance(model.search, optuna.Study)
}
cv_summary = pd.DataFrame(
    [
        {"Model": name, "CV macro F1": model.cv_mean, "CV macro F1 std": model.cv_std}
        for name, model in fitted.items()
    ]
).set_index("Model")
selected_model_name = next(iter(fitted))

print(f"Selected by mean CV macro F1: {selected_model_name}")
display(
    pd.Series(best_params_by_model, name="Refit parameters")
    .loc[cv_summary.index]
    .to_frame()
)

## 4.2. Search progress

Dots show the mean validation macro F1 for each trial. The line tracks the best score found so far within each ensemble family.

In [ ]:
fig, axes = plt.subplots(
    2, 2, figsize=(12, 7), sharex=True, sharey=True, layout="constrained"
)
for (name, study), ax in zip(studies.items(), axes.flat):
    trial_scores = np.array([trial.value for trial in study.trials])
    trial_numbers = np.arange(1, len(trial_scores) + 1)
    ax.scatter(
        trial_numbers, trial_scores, color="#557b9b", alpha=0.55, label="Trial average"
    )
    ax.step(
        trial_numbers,
        np.maximum.accumulate(trial_scores),
        where="post",
        color="#287969",
        linewidth=2,
        label="Best so far",
    )
    ax.set_title(name)
axes[0, 0].legend(loc="lower right")
fig.supxlabel("Optuna trial")
fig.supylabel("Mean CV macro F1")
plt.show()

## 4.3. Baseline grid results

The logistic regression curves show the variation across folds for each regularization strength and class-weight setting. The decision tree heatmaps show the mean score for each depth and leaf-size combination.

In [ ]:
logistic_results = pd.DataFrame(baseline_searches["Logistic regression"].cv_results_)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True, layout="constrained")
for (_, results), ax in zip(
    logistic_results.groupby(
        "param_classifier__class_weight", dropna=False, sort=False
    ),
    axes,
):
    c = results["param_classifier__C"].to_numpy(dtype=float)
    mean = results["mean_test_score"].to_numpy()
    std = results["std_test_score"].to_numpy()
    weight = results["param_classifier__class_weight"].iloc[0]
    ax.plot(c, mean, "o-", color="#287969")
    ax.fill_between(c, mean - std, mean + std, color="#287969", alpha=0.18)
    ax.set(xscale="log", xlabel="C", title=f"Class weights · {weight or 'uniform'}")
axes[0].set_ylabel("Mean CV macro F1 ± 1 SD")
fig.suptitle("Logistic regression")
plt.show()

In [ ]:
# Plot validation macro F1 for every tree depth, leaf size, and class-weight combination.
from matplotlib.patches import Rectangle

tree_search = baseline_searches["Decision tree"]

tree_validation_results = pd.DataFrame(tree_search.cv_results_)
tree_validation_results["Depth label"] = tree_validation_results[
    "param_classifier__max_depth"
].map(lambda depth: "Unlimited" if pd.isna(depth) else str(int(depth)))
tree_weight_values = tree_search.param_grid["classifier__class_weight"]
depth_labels = [
    "Unlimited" if depth is None else str(depth)
    for depth in tree_search.param_grid["classifier__max_depth"]
]
leaf_values = tree_search.param_grid["classifier__min_samples_leaf"]
validation_min = tree_validation_results["mean_test_score"].min()
validation_max = tree_validation_results["mean_test_score"].max()

fig, axes = plt.subplots(1, len(tree_weight_values), figsize=(12, 5.3))
for class_weight, ax in zip(tree_weight_values, axes):
    weight_mask = tree_validation_results["param_classifier__class_weight"].isna()
    if class_weight is not None:
        weight_mask = tree_validation_results["param_classifier__class_weight"].eq(
            class_weight
        )
    scores = (
        tree_validation_results.loc[weight_mask]
        .pivot(
            index="Depth label",
            columns="param_classifier__min_samples_leaf",
            values="mean_test_score",
        )
        .reindex(index=depth_labels, columns=leaf_values)
    )
    sns.heatmap(
        scores,
        annot=True,
        fmt=".4f",
        cmap="YlGnBu",
        vmin=validation_min,
        vmax=validation_max,
        linewidths=0.5,
        cbar_kws={"label": "Mean validation macro F1"},
        ax=ax,
    )
    if class_weight == tree_search.best_params_["classifier__class_weight"]:
        selected_depth = tree_search.best_params_["classifier__max_depth"]
        selected_depth_label = (
            "Unlimited" if selected_depth is None else str(selected_depth)
        )
        selected_leaf_size = tree_search.best_params_["classifier__min_samples_leaf"]
        selected_column = leaf_values.index(selected_leaf_size)
        selected_row = depth_labels.index(selected_depth_label)
        ax.add_patch(
            Rectangle(
                (selected_column, selected_row),
                1,
                1,
                fill=False,
                edgecolor="#e4b348",
                linewidth=3,
            )
        )
    ax.set(
        title=f"class_weight = {class_weight}",
        xlabel="min_samples_leaf",
        ylabel="max_depth",
    )
    ax.tick_params(axis="y", labelrotation=0)

fig.suptitle("Decision tree | validation macro F1 across the search grid", y=0.995)
fig.text(
    0.5,
    0.01,
    "Gold border marks the configuration selected by mean validation macro F1.",
    ha="center",
    fontsize=10,
)
plt.tight_layout(rect=(0, 0.04, 1, 0.95))
plt.show()

selected_tree_std = tree_search.cv_results_["std_test_score"][tree_search.best_index_]
print(
    f"Selected tree: max_depth={tree_search.best_params_['classifier__max_depth']}, "
    f"min_samples_leaf={tree_search.best_params_['classifier__min_samples_leaf']}, "
    f"class_weight={tree_search.best_params_['classifier__class_weight']}. "
    f"Validation macro F1 = {tree_search.best_score_:.4f} ± {selected_tree_std:.4f} "
    "(mean ± standard deviation across folds)."
)

# 5. Holdout evaluation

The model with the highest mean cross-validation macro F1 was selected on train. The chart pairs that CV estimate with the single holdout score for each fitted model. Dots show individual validation folds, and whiskers show one standard deviation across folds.

In [ ]:
X_test, test_targets = extract_inputs(holdout)

In [ ]:
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

class_codes = np.arange(len(CLASS_ORDER))
test_predictions = {name: model.predict(X_test) for name, model in models.items()}
test_macro_f1 = pd.Series(
    {
        name: f1_score(
            test_targets,
            predictions,
            labels=class_codes,
            average="macro",
            zero_division=0,
        )
        for name, predictions in test_predictions.items()
    },
    name="Holdout macro F1",
)
comparison = cv_summary.join(test_macro_f1)
comparison["Selected by CV"] = comparison.index == selected_model_name
metric_columns = ["CV macro F1", "CV macro F1 std", "Holdout macro F1"]
display(comparison[[*metric_columns, "Selected by CV"]].round(4))

In [ ]:
model_order = comparison.index.to_list()
model_positions = np.arange(len(model_order))
model_labels = [
    f"{name} (selected)" if name == selected_model_name else name
    for name in model_order
]
model_colors = [
    "#237769" if name == selected_model_name else "#a3b8c5" for name in model_order
]

fig, (cv_ax, test_ax) = plt.subplots(
    1, 2, sharey=True, figsize=(12, 5), layout="constrained"
)
cv_ax.barh(model_positions, comparison["CV macro F1"], color=model_colors, height=0.59)
cv_ax.errorbar(
    comparison["CV macro F1"],
    model_positions,
    xerr=comparison["CV macro F1 std"],
    fmt="none",
    ecolor="#192f3a",
    capsize=4,
    zorder=3,
)
for position, name in enumerate(model_order):
    fold_scores = cv_scores_by_model[name]
    cv_ax.scatter(
        fold_scores,
        position + np.linspace(-0.19, 0.19, len(fold_scores)),
        s=19,
        facecolor="white",
        edgecolor="#35586b",
        linewidth=0.8,
        zorder=4,
    )

holdout_bars = test_ax.barh(
    model_positions, comparison["Holdout macro F1"], color=model_colors, height=0.59
)
test_ax.bar_label(holdout_bars, fmt="%.3f", padding=5)
cv_ax.set_yticks(model_positions, labels=model_labels)
cv_ax.invert_yaxis()
cv_ax.set(title="Train cross-validation", xlabel="Mean validation macro F1")
test_ax.set(title="Holdout", xlabel="Macro F1")
for ax in (cv_ax, test_ax):
    ax.set_xlim(0, 1)
    ax.grid(axis="y", visible=False)
plt.show()

## 5.1. Selected model by class

The confusion matrices show counts and the share within each actual class. The error table ranks misclassifications by count and shows the corresponding within-class rate.

In [ ]:
test_confusions = {
    name: confusion_matrix(test_targets, predictions, labels=class_codes)
    for name, predictions in test_predictions.items()
}
selected_predictions = test_predictions[selected_model_name]
selected_confusion = test_confusions[selected_model_name]
selected_weighted_f1 = f1_score(
    test_targets,
    selected_predictions,
    labels=class_codes,
    average="weighted",
    zero_division=0,
)
selected_balanced_accuracy = balanced_accuracy_score(test_targets, selected_predictions)
print(selected_model_name)
print(
    classification_report(
        test_targets,
        selected_predictions,
        labels=class_codes,
        target_names=CLASS_ORDER,
        digits=4,
        zero_division=0,
    )
)
print(f"Weighted F1 = {selected_weighted_f1:.4f}")
print(f"Balanced accuracy = {selected_balanced_accuracy:.4f}")

class_totals = selected_confusion.sum(axis=1, keepdims=True)
confusion_rates = selected_confusion / class_totals
fig, axes = plt.subplots(1, 2, figsize=(10, 4.6), layout="constrained")
for ax, matrix, title, number_format in zip(
    axes,
    (selected_confusion, confusion_rates),
    ("Student counts", "Share of actual class"),
    ("d", ".1%"),
):
    ConfusionMatrixDisplay(matrix, display_labels=CLASS_ORDER).plot(
        ax=ax,
        cmap="Blues",
        colorbar=False,
        values_format=number_format,
    )
    ax.set_title(title)
    ax.grid(visible=False)
fig.suptitle(f"{selected_model_name} on holdout")
plt.show()

confusion_errors = (
    pd.DataFrame(
        [
            {
                "Actual class": actual,
                "Predicted class": predicted,
                "Students": selected_confusion[row, column],
                "Share of actual class, %": 100 * confusion_rates[row, column],
            }
            for row, actual in enumerate(CLASS_ORDER)
            for column, predicted in enumerate(CLASS_ORDER)
            if row != column
        ]
    )
    .sort_values("Students", ascending=False, kind="stable")
    .reset_index(drop=True)
)
display(confusion_errors.round(1))

## 5.2. Dropout flags

The default three-class decision rule issues a flag when the predicted class is `Dropout`. Recall measures how many of the actual `Dropout` students the model finds. Low recall leaves more students without early outreach. Precision measures how many flagged students actually have the `Dropout` outcome. Low precision can create unnecessary outreach and take up staff time. We report precision as zero for a model that issues no flags.

`Enrolled` is the observed status at the study endpoint. Some of these students may still benefit from a support offer.

In [ ]:
dropout_code = CLASS_ORDER.index("Dropout")
dropout_rows = []
for name in model_order:
    matrix = test_confusions[name]
    true_positives = matrix[dropout_code, dropout_code]
    actual_dropouts = matrix[dropout_code, :].sum()
    flagged = matrix[:, dropout_code].sum()
    dropout_rows.append(
        {
            "Model": name,
            "Dropout precision": true_positives / flagged if flagged else 0.0,
            "Dropout recall": true_positives / actual_dropouts,
            "Correctly flagged": true_positives,
            "Missed Dropout": actual_dropouts - true_positives,
            "Other outcomes flagged": flagged - true_positives,
        }
    )
dropout_metrics = pd.DataFrame(dropout_rows).set_index("Model")
display(dropout_metrics.round(4))

fig, ax = plt.subplots(figsize=(9, 5), layout="constrained")
for offset, metric, color in [
    (-0.18, "Dropout precision", "#476887"),
    (0.18, "Dropout recall", "#228172"),
]:
    bars = ax.barh(
        model_positions + offset,
        dropout_metrics[metric],
        height=0.33,
        color=color,
        label=metric,
    )
    ax.bar_label(bars, fmt="%.2f", padding=4)
ax.set_yticks(model_positions, labels=model_labels)
ax.invert_yaxis()
ax.set(xlim=(0, 1), xlabel="Score", title="Precision and recall for Dropout on holdout")
ax.legend(loc="lower right")
ax.grid(axis="y", visible=False)
plt.show()

# 6. Enrollment features used by the ensembles

The plots show the highest-ranked original features for each ensemble. Each categorical feature receives the sum of the importances of its one-hot columns. Random forest and gradient boosting report impurity-based importance, XGBoost reports total gain, and LightGBM reports gain. We convert each model’s importances to shares of its own total so the feature ranks are easier to read.

These scores describe the trees’ training splits. Correlated features may share importance, and fields with many possible splits can receive larger scores. Establishing causality requires a separate study. Decisions about an individual student require context beyond the model output.

In [ ]:
def original_feature_importances(model):
    preprocessor = model.named_steps["preprocessor"]
    classifier = model.named_steps["classifier"]
    encoder = preprocessor.named_transformers_["categorical"].named_steps["encoder"]
    model_importance = classifier.feature_importances_
    input_features = np.empty(model_importance.size, dtype=object)
    input_features[preprocessor.output_indices_["numeric"]] = numeric_features
    input_features[preprocessor.output_indices_["categorical"]] = np.repeat(
        categorical_features,
        [len(categories) for categories in encoder.categories_],
    )
    importance = pd.Series(model_importance, index=input_features)
    importance = importance.groupby(level=0).sum().reindex(ALLOWED_FEATURES)
    return 100 * importance / importance.sum()


ensemble_importance = pd.DataFrame(
    {name: original_feature_importances(models[name]) for name in studies}
).rename_axis("Enrollment feature")
fig, axes = plt.subplots(2, 2, figsize=(13, 8), layout="constrained")
importance_limit = ensemble_importance.max().max() * 1.2
for ax, name in zip(axes.flat, ensemble_importance.columns):
    top_features = ensemble_importance[name].nlargest(7).sort_values()
    bars = ax.barh(top_features.index, top_features, color="#4b8b83")
    ax.bar_label(bars, fmt="%.1f%%", padding=4)
    ax.set(
        title=name, xlabel="Share of model importance, %", xlim=(0, importance_limit)
    )
    ax.grid(axis="y", visible=False)
fig.suptitle("Top enrollment features by ensemble")
plt.show()

# 7. Experiment conclusions

In [ ]:
from IPython.display import Markdown

winner = comparison.loc[selected_model_name]
runner_up_name = comparison.index[1]
runner_up = comparison.loc[runner_up_name]
selected_dropout = dropout_metrics.loc[selected_model_name]
recall_alternative_name = dropout_metrics.drop(index=selected_model_name)[
    "Dropout recall"
].idxmax()
recall_alternative = dropout_metrics.loc[recall_alternative_name]
most_frequent_error = confusion_errors.iloc[0]
leading_features = ", ".join(
    f"**{feature}** ({name})" for name, feature in ensemble_importance.idxmax().items()
)
test_leader = test_macro_f1.idxmax()

summary = f"""
**Model for early support.** Use **{selected_model_name}** to prioritize offers of voluntary support
at the end of enrollment, with staff reviewing the student’s context. Its mean training CV macro F1
was **{winner["CV macro F1"]:.4f} ± {winner["CV macro F1 std"]:.4f}** across folds.
**{runner_up_name}** was second on CV with **{runner_up["CV macro F1"]:.4f}** mean macro F1.

**Holdout results.** The selected model achieved **{winner["Holdout macro F1"]:.4f}** macro F1,
**{selected_weighted_f1:.4f}** weighted F1 and **{selected_balanced_accuracy:.4f}** balanced accuracy.
The highest observed holdout macro F1 was **{test_macro_f1.loc[test_leader]:.4f}** ({test_leader}).
The most frequent error was predicting `{most_frequent_error["Predicted class"]}` for
`{most_frequent_error["Actual class"]}` students (**{most_frequent_error["Students"]:,}** students,
**{most_frequent_error["Share of actual class, %"]:.1f}%** of that actual class).

**Trade-off for Dropout.** Recall was **{selected_dropout["Dropout recall"]:.1%}**. The model
found **{int(selected_dropout["Correctly flagged"]):,}** students with the `Dropout` outcome
and missed **{int(selected_dropout["Missed Dropout"]):,}**.
Precision was **{selected_dropout["Dropout precision"]:.1%}** with
**{int(selected_dropout["Other outcomes flagged"]):,}** flags for `Enrolled` or `Graduate` students.
Among the other models, **{recall_alternative_name}** had the highest `Dropout` recall
(**{recall_alternative["Dropout recall"]:.1%}**). It found
**{int(recall_alternative["Correctly flagged"]):,}** `Dropout` students and flagged
**{int(recall_alternative["Other outcomes flagged"]):,}** students with other outcomes.

**Feature importance.** The top-ranked features were {leading_features}.
"""
display(Markdown(summary))